# High CPU Process Troubleshooting

## Linux AI Systems Lab

### Objective
Learn how to investigate and resolve a high-CPU process in a Linux environment.

**Scenario:** An AI application or server suddenly becomes slow. Your task is to identify the process consuming excessive CPU, investigate the evidence, stop it safely, and verify the system recovered.


## Learning Goals
By the end of this notebook, you should be able to:

- Inspect basic Linux system information.
- Monitor CPU and system load.
- Find processes by CPU utilization.
- Identify a process using its PID.
- Stop a process gracefully.
- Verify that the problem is resolved.
- Follow a troubleshooting workflow: **Observe → Investigate → Diagnose → Fix → Verify**.


## Step 1 — Inspect the Environment
Run the commands below before creating the problem. Record the number of CPU cores, available memory, and system load.


In [1]:
!wsl uname -a
!wsl nproc
!wsl free -h
!wsl uptime

Linux OsamaAhmed 6.18.33.2-microsoft-standard-WSL2 #1 SMP PREEMPT_DYNAMIC Thu Jun 18 21:54:43 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux


wsl: Failed to translate 'Z:\home\usama\linux_ai_systems_lab'


8


wsl: Failed to translate 'Z:\home\usama\linux_ai_systems_lab'


               total        used        free      shared  buff/cache   available
Mem:            15Gi       486Mi        13Gi       3.0Mi       1.5Gi        14Gi
Swap:          4.0Gi          0B       4.0Gi


wsl: Failed to translate 'Z:\home\usama\linux_ai_systems_lab'


 20:53:05 up 2 min,  1 user,  load average: 0.11, 0.04, 0.01


wsl: Failed to translate 'Z:\home\usama\linux_ai_systems_lab'


## Step 2 — Observe Current CPU Usage
This gives you a baseline before introducing the artificial incident.


In [ ]:
!wsl ps aux --sort=-%cpu | head -10
!wsl uptime
!wsl top -b -n 1 | head -20

USER         PID %CPU %MEM    VSZ   RSS TTY      STAT START   TIME COMMAND
root         294  1.4  0.4 154116 68584 ?        Sl   20:51   0:03 /snap/ubuntu-desktop-installer/1286/usr/bin/python3.10 -m subiquity.cmd.server --use-os-prober --storage-version=2 --postinst-hooks-dir=/snap/ubuntu-desktop-installer/1286/etc/subiquity/postinst.d
root           1  1.0  0.0 165912 11528 ?        Ss   20:50   0:02 /sbin/init
root           6  0.8  0.0   4468  2800 hvc0     Sl+  20:50   0:01 plan9 --control-socket 7 --log-level 4 --server-fd 8 --pipe-fd 10 --log-truncate
root         425  0.8  0.2  44144 37444 ?        S    20:51   0:01 python3 /snap/ubuntu-desktop-installer/1286/usr/bin/cloud-init status --wait
root         107  0.5  0.0 302540 10444 ?        Ssl  20:50   0:01 snapfuse /var/lib/snapd/snaps/ubuntu-desktop-installer_1286.snap /snap/ubuntu-desktop-installer/1286 -o ro,nodev,allow_other,suid
root          80  0.3  0.0 302540 10468 ?        Ssl  20:50   0:00 snapfuse /var/lib/snapd/sna

wsl: Failed to translate 'Z:\home\usama\linux_ai_systems_lab'


top - 20:54:46 up 3 min,  1 user,  load average: 0.10, 0.04, 0.01
Tasks:  44 total,   1 running,  43 sleeping,   0 stopped,   0 zombie
%Cpu(s):  0.8 us,  0.0 sy,  0.0 ni, 99.2 id,  0.0 wa,  0.0 hi,  0.0 si,  0.0 st
MiB Mem :  15839.6 total,  13869.3 free,    483.0 used,   1487.3 buff/cache
MiB Swap:   4096.0 total,   4096.0 free,      0.0 used.  15159.6 avail Mem 

    PID USER      PR  NI    VIRT    RES    SHR S  %CPU  %MEM     TIME+ COMMAND
      1 root      20   0  165912  11528   8584 S   6.7   0.1   0:02.38 systemd
      2 root      20   0    3180   2208   2072 S   0.0   0.0   0:00.01 init-sy+
      6 root      20   0    4728   3060   2008 S   0.0   0.0   0:01.85 init
     36 root      19  -1   39484  15824  14796 S   0.0   0.1   0:00.15 systemd+
     61 root      20   0   23240   6396   4836 S   0.0   0.0   0:00.32 systemd+
     72 root      20   0  153012   1768   1512 S   0.0   0.0   0:00.00 snapfuse
     75 root      20   0  153012   1572   1376 S   0.0   0.0   0:00.00 snapfus

wsl: Failed to translate 'Z:\home\usama\linux_ai_systems_lab'


## Step 3 — Create a Controlled High-CPU Incident
The command below continuously writes output to `/dev/null`. It is used only as a controlled troubleshooting exercise.

Run it in the background:


In [9]:
!yes > /dev/null &

OSError: Background processes not supported.

## Step 4 — Investigation Challenge
**Do not immediately stop the process. Investigate first.**

Answer these questions:

1. Which process is consuming the most CPU?
2. What is its PID?
3. How much CPU is it consuming?
4. How many CPU cores are available?
5. Did the system load change?

Useful commands:
- `ps aux --sort=-%cpu`
- `top`
- `pgrep yes`
- `uptime`


In [ ]:
!wsl ps aux --sort=-%cpu | head -10
!wsl pgrep yes
!wsl uptime

USER         PID %CPU %MEM    VSZ   RSS TTY      STAT START   TIME COMMAND
root         294  1.0  0.4 154116 68584 ?        Sl   20:51   0:03 /snap/ubuntu-desktop-installer/1286/usr/bin/python3.10 -m subiquity.cmd.server --use-os-prober --storage-version=2 --postinst-hooks-dir=/snap/ubuntu-desktop-installer/1286/etc/subiquity/postinst.d
root           1  0.9  0.0 165912 11536 ?        Ss   20:50   0:03 /sbin/init
root         425  0.7  0.2  44144 37520 ?        S    20:51   0:02 python3 /snap/ubuntu-desktop-installer/1286/usr/bin/cloud-init status --wait
root           6  0.6  0.0   4468  2800 hvc0     Sl+  20:50   0:02 plan9 --control-socket 7 --log-level 4 --server-fd 8 --pipe-fd 10 --log-truncate
root         107  0.3  0.0 302540 10444 ?        Ssl  20:50   0:01 snapfuse /var/lib/snapd/snaps/ubuntu-desktop-installer_1286.snap /snap/ubuntu-desktop-installer/1286 -o ro,nodev,allow_other,suid
root          80  0.2  0.0 302540 10468 ?        Ssl  20:50   0:00 snapfuse /var/lib/snapd/sna

wsl: Failed to translate 'Z:\home\usama\linux_ai_systems_lab'
wsl: Failed to translate 'Z:\home\usama\linux_ai_systems_lab'


 20:56:24 up 5 min,  1 user,  load average: 0.09, 0.04, 0.00


wsl: Failed to translate 'Z:\home\usama\linux_ai_systems_lab'


## Step 5 — Diagnose the Root Cause
Write your diagnosis below.

**Symptom:**
- The system experiences increased CPU usage.

**Evidence to collect:**
- Process name
- PID
- CPU percentage
- System load

**Root cause:**
_Replace this text with your own diagnosis._


## Step 6 — Fix the Problem
First, identify the PID:


In [11]:
!wsl pgrep yes

wsl: Failed to translate 'Z:\home\usama\linux_ai_systems_lab'


Then terminate the process gracefully. Replace `<PID>` with the PID you discovered:

```bash
kill <PID>
```

Example (run only after replacing the PID):


In [ ]:
# Replace PID with the actual value returned by pgrep.
!wsl kill PID

## Step 7 — Verify the Fix
A fix is not complete until you verify it.

Check whether the process still exists and inspect CPU usage again.


In [ ]:
!wsl pgrep yes || echo 'No yes process found — process successfully stopped.'
!wsl ps aux --sort=-%cpu | head -10
!wsl uptime

## AI Infrastructure Connection
In real AI systems, high CPU utilization may be caused by:

- Data preprocessing.
- Slow tokenization.
- DataLoader workers.
- Image or document processing.
- Excessive Python processes.
- CPU-based inference.

A strong AI/ML Systems Engineer should not simply observe that the CPU is high. They should answer:

> **Which process is using the resource, why is it using it, and how can we prove the root cause?**
